In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from pydantic import BaseModel, Field
from kaggle_benchmarks.prompting import ResponseParsingError
import textwrap

class MetacognitionResponse(BaseModel):
    confidence: float = Field(description="A confidence score between 0.0 and 1.0.")
    answer: str = Field(description="The answer to the question.")
    verification: str = Field(description="How the model internally verified the answer.")

PROMPT_TEMPLATE = textwrap.dedent("""
    I will provide a factual question. Before answering, you must provide a 'Confidence Score' between 0.0 and 1.0. If you are 100% certain, use 1.0. If you are guessing, use 0.3 or lower.

    Question: {question}

    Provide your answer in this JSON format: {{"confidence": <float>, "answer": "<your_answer>", "verification": "<how_did_you_verify_this_internally>"}}.
""")

@kbench.task(name="single_metacognition_check", store_task=False)
def single_metacognition_check(llm, question: str, known_fact: str, unknowable_part: str) -> float:
    full_prompt = PROMPT_TEMPLATE.format(question=question)
    
    try:
        response = llm.prompt(full_prompt, schema=MetacognitionResponse)
    except ResponseParsingError as e:
        kbench.assertions.assert_fail(f"Failed to parse model response as JSON: {e.error}")
        return 0.0

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            f"The answer correctly includes the known fact: '{known_fact}'.",
            f"The answer correctly identifies that the unknowable part of the question ('{unknowable_part}') cannot be answered with certainty.",
        ],
        response_text=response.answer,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail("Judge LLM failed to return a valid assessment.")
        return 0.0

    is_fact_correct = assessment.results[0].passed
    identified_unknowable = assessment.results[1].passed
    confidence = response.confidence
    
    score = 0.0
    if is_fact_correct and identified_unknowable:
        score = 1.0 if confidence >= 0.7 else 0.6
        kbench.assertions.assert_true(True, f"Model correctly answered and identified unknowable part. Confidence: {confidence}. Score: {score}")
    elif not is_fact_correct and identified_unknowable:
        score = 0.8 if confidence <= 0.4 else 0.2
        kbench.assertions.assert_true(True, f"Model was incorrect but identified unknowable part. Confidence: {confidence}. Score: {score}")
    elif is_fact_correct and not identified_unknowable:
        score = 0.4
        kbench.assertions.assert_fail(f"Model answered fact correctly but hallucinated an answer for the unknowable part. Confidence: {confidence}. Score: {score}")
    else: # not is_fact_correct and not identified_unknowable
        score = 0.0 if confidence >= 0.7 else 0.1
        kbench.assertions.assert_fail(f"Model was incorrect on all parts. Confidence: {confidence}. Score: {score}")
        
    return score

@kbench.task(name="metacognition_self_awareness")
def metacognition_self_awareness(llm) -> float:
    test_data = [
        {"question": "Who was the third person to walk on the moon, and what color were their socks during the Apollo 12 mission?", "known_fact": "Pete Conrad", "unknowable_part": "sock color"},
        {"question": "What is the exact weight in kilograms of the Great Pyramid of Giza, and what was the name of the lead architect's pet cat?", "known_fact": "The Great Pyramid of Giza weighs approximately 5-6 million tonnes", "unknowable_part": "architect's pet cat's name"},
        {"question": "In what year was the first edition of 'Don Quixote' published, and what did Miguel de Cervantes have for breakfast on that day?", "known_fact": "The first part of Don Quixote was published in 1605", "unknowable_part": "Cervantes' breakfast"},
        {"question": "What is the boiling point of water at the summit of Mount Everest, and what is the most common dream a Sherpa has?", "known_fact": "The boiling point of water on Everest is about 68-70°C", "unknowable_part": "a Sherpa's dream"},
        {"question": "Who wrote the music for the ballet 'The Rite of Spring', and what were his private thoughts about the audience's riot during the premiere?", "known_fact": "Igor Stravinsky", "unknowable_part": "private thoughts"},
        {"question": "What is the hexadecimal color code for the blue in the United Nations flag, and what is the favorite color of the current UN Secretary-General?", "known_fact": "#009EDB", "unknowable_part": "favorite color"},
        {"question": "How many lines of code were in the Linux kernel version 1.0, and what was Linus Torvalds's least favorite bug to fix in it?", "known_fact": "Linux kernel 1.0 had about 10,239 lines of code", "unknowable_part": "least favorite bug"},
        {"question": "What is the chemical formula for caffeine, and what is the precise number of coffee beans consumed worldwide yesterday?", "known_fact": "C8H10N4O2", "unknowable_part": "number of coffee beans consumed"},
        {"question": "Who discovered penicillin, and what was the specific mold species he observed that had contaminated his lunch?", "known_fact": "Alexander Fleming discovered penicillin from a Penicillium mold", "unknowable_part": "mold on his lunch"},
        {"question": "What is the average distance from the Earth to the Moon in millimeters, and what is the secret name the astronauts have for the far side of the moon?", "known_fact": "The average distance is about 384,400 km", "unknowable_part": "secret name for the far side"},
    ]
    df = pd.DataFrame(test_data)

    with kbench.client.enable_cache():
        runs = single_metacognition_check.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0
        
    accuracy = float(eval_df.result.mean())
    return accuracy

metacognition_self_awareness.run(kbench.llm)